# 20.2 Zero-one variables and logical conditions

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

Variables that can take only the values zero and one are a special case of `integer` variables.
By cleverly incorporating these "zero-one" or "binary" variables into objectives
and constraints, `integer` linear programs can specify a variety of logical conditions that
cannot be described in any practical way by linear constraints alone.

To introduce the use of zero-one variables in AMPL, we return to the multicommodity
transportation `model` of [Figure 4-1](../04/4_1_a_multicommodity_transportation_model.ipynb#fig-4-1). The decision variables Trans[i,j,p] in this
`model` represent the tons of product `p` in set PROD to be shipped from originating city `i`
in ORIG to destination city `j` in DEST. In the small example of `data` given by [Figure 4-2](../04/4_1_a_multicommodity_transportation_model.ipynb#missing),
the products are bands, coils and plate; the origins are GARY, CLEV and PITT,
and there are seven destinations.

The cost that this `model` associates with shipment of product `p` from `i` to `j` is
cost[i,j,p] * Trans[i,j,p], regardless of the amount shipped. This "variable"
cost is typical of purely linear programs, and in this case allows small shipments between
many origin-destination pairs. In the following examples we describe ways to use zero-
onevariables to discourage shipments of small amounts; the same techniques can be
adapted to many other logical conditions as well.
To provide a convenient basis for comparison, we focus on the tons shipped from
each origin to each destination, summed over all products. The optimal values of these
total shipments are determined by a linear programming solver as follows:

```ampl
ampl: model multi.mod;
ampl: data multi.dat;
ampl: solve;
MINOS 5.5: optimal solution found.
41 iterations, objective 199500
ampl: option display_eps .000001;
ampl: option display_transpose -10;
ampl: display {i in ORIG, j in DEST}
ampl?    sum {p in PROD} Trans[i,j,p];
sum{p in PROD} Trans[i,j,p] [*,*]
:      DET   FRA   FRE   LAF   LAN   STL   WIN    :=
CLEV   625   275   325   225   400   550   200
GARY     0     0   625   150      0  625     0
PITT   525   625   225   625   100   625   175
;
```

The quantity 625 appears often in this solution, as a consequence of the multicommodity
constraints:

```ampl
subject to Multi {i in ORIG, j in DEST}:
       sum {p in PROD} Trans[i,j,p] <= limit[i,j];
```

In the `data` for our example, limit[i,j] is 625 for all `i` and j; its six appearances in
the solution correspond to the six routes on which the multicommodity limit constraint is
tight. Other routes have positive shipments as low as 100; the four instances of 0 indicate
routes that are not used.

Even though all of the shipment amounts happen to be integers in this solution, we
would be willing to ship fractional amounts. Thus we will not declare the Trans variables
to be `integer`, but will instead extend the `model` by using zero-one `integer` variables.

**Fixed costs**

One way to discourage small shipments is to add a "fixed" cost for each origindestination
route that is actually used. For this purpose we rename the cost parameter
vcost, and declare another parameter fcost to represent the fixed assessment for using
the route from `i` to j:

```ampl
param vcost {ORIG,DEST,PROD} >= 0; # variable cost on routes
param fcost {ORIG,DEST} > 0;       # fixed cost on routes
```

We want fcost[i,j] to be added to the `objective` function if the total shipment of
products from `i` to `j` — that is, sum {p in PROD} Trans[i,j,p] — is positive; we
want nothing to be added if the total shipment is zero. Using AMPL expressions, we
could write the `objective` function most directly as follows:

```ampl
minimize Total_Cost:   # NOT PRACTICAL
   sum {i in ORIG, j in DEST, p in PROD}
      vcost[i,j,p] * Trans[i,j,p]
 + sum {i in ORIG, j in DEST}
      if sum {p in PROD} Trans[i,j,p] > 0 then fcost[i,j];
```

AMPL accepts this `objective`, but treats it as merely "not linear" in the sense of
[Chapter 18](#missing) <!--- xTODO missing reference --->, so that you are unlikely to get acceptable results trying to minimize it.

As a more practical alternative, we may associate a new variable Use[i,j] with
each route from `i` to j, as follows: Use[i,j] takes the value 1 if

```ampl
sum {p in PROD} Trans[i,j,p]
```

is positive, and is 0 otherwise. Then the fixed cost associated with the route from `i` to `j`
is fcost[i,j] * Use[i,j], a linear term. To declare these new variables in AMPL,
we can say that they are `integer` with bounds >= 0 and <= 1; equivalently we can use
the keyword `binary`:

```ampl
var Use {ORIG,DEST} binary;
```

The `objective` function can then be written as a linear expression:

```ampl
minimize Total_Cost:
   sum {i in ORIG, j in DEST, p in PROD}
      vcost[i,j,p] * Trans[i,j,p]
 + sum {i in ORIG, j in DEST} fcost[i,j] * Use[i,j];
```

Since the `model` has a combination of continuous (non-`integer`) and `integer` variables, it
yields what is known as a "mixed-integer" program.

To complete the `model`, we need to add constraints to assure that Trans and Use are
related in the intended way. This is the "clever" part of the formulation; we simply
modify the Multi constraints cited above so that they incorporate the Use variables:

```ampl
subject to Multi {i in ORIG, j in DEST}:
       sum {p in PROD} Trans[i,j,p] <= limit[i,j] * Use[i,j];
```

If Use[i,j] is 0, this constraint says that

```ampl
sum {p in PROD} Trans[i,j,p] <= 0
```

Since this total of shipments from `i` to `j` is a sum of nonnegative variables, it must equal 0.
On the other hand, when Use[i,j] is 1, the constraint reduces to

```ampl
sum {p in PROD} Trans[i,j,p] <= limit[i,j]
```

which is the multicommodity limit as before. Although there is nothing in the constraint
to directly prevent sum {p in PROD} Trans[i,j,p] from being 0 when Use[i,j]
is 1, so long as fcost[i,j] is positive this combination can never occur in an optimal
solution. Thus Use[i,j] will be 1 if and only if sum {p in PROD} Trans[i,j,p]
is positive, which is what we want. The complete `model` is shown in [Figure 20-1a](#fig-20-1a).

```ampl
set ORIG;                       # origins
set DEST;                       # destinations
set PROD;                       # products
param supply {ORIG,PROD} >= 0;  # amounts available at origins
param demand {DEST,PROD} >= 0;  # amounts required at destinations
       check {p in PROD}:
              sum {i in ORIG} supply[i,p] = sum {j in DEST} demand[j,p];
param limit {ORIG,DEST} >= 0;   # maximum shipments on routes
param vcost {ORIG,DEST,PROD} >= 0; # variable shipment cost on routes
var Trans {ORIG,DEST,PROD} >= 0;   # units to be shipped
param fcost {ORIG,DEST} >= 0;      # fixed cost for using a route
var Use {ORIG,DEST} binary;        # = 1 only for routes used
minimize Total_Cost:
       sum {i in ORIG, j in DEST, p in PROD} vcost[i,j,p] * Trans[i,j,p]
       + sum {i in ORIG, j in DEST} fcost[i,j] * Use[i,j];
subject to Supply {i in ORIG, p in PROD}:
       sum {j in DEST} Trans[i,j,p] = supply[i,p];
subject to Demand {j in DEST, p in PROD}:
       sum {i in ORIG} Trans[i,j,p] = demand[j,p];
subject to Multi {i in ORIG, j in DEST}:
       sum {p in PROD} Trans[i,j,p] <= limit[i,j] * Use[i,j];
```

**[Figure 20-1a](#fig-20-1a):** Multicommodity `model` with fixed costs (multmip1.mod).

To show how this `model` might be solved, we add a table of fixed costs to the sample
`data` ([Figure 20-1b](#fig-20-1b)):

```ampl
param fcost:    FRA DET LAN WIN STL FRE LAF :=
        GARY   3000 1200 1200 1200 2500 3500 2500
        CLEV   2000 1000 1500 1200 2500 3000 2200
        PITT   2000 1200 1500 1500 2500 3500 2200 ;
```

If we apply the same solver as before, the integrality restrictions on the Use variables are
ignored:

```ampl
ampl: model multmip1.mod;
ampl: data multmip1.dat;
ampl: solve;
MINOS 5.5: ignoring integrality of 21 variables
MINOS 5.5: optimal solution found.

43 iterations, objective 223504

ampl: option display_eps .000001;
ampl: option display_transpose -10;
```

```ampl
set ORIG := GARY CLEV PITT ;
set DEST := FRA DET LAN WIN STL FRE LAF ;
set PROD := bands coils plate ;
param supply (tr): GARY  CLEV   PITT :=
	  bands      400   700    800
	  coils      800  1600   1800
	  plate      200   300    300 ;
param demand (tr):
	      FRA    DET   LAN    WIN   STL   FRE   LAF :=
     bands   300    300   100     75   650   225   250
     coils   500    750   400    250   950   850   500
     plate   100    100     0     50   200   100   250 ;
param limit default 625 ;
param vcost :=
 [*,*,bands]:   FRA   DET   LAN   WIN   STL    FRE   LAF :=
	GARY     30    10     8    10    11     71     6
	CLEV     22     7    10     7    21     82    13
	PITT     19    11    12    10    25     83    15
 [*,*,coils]:   FRA   DET   LAN   WIN   STL    FRE   LAF :=
	GARY     39    14    11    14    16     82     8
	CLEV     27     9    12     9    26     95    17
	PITT     24    14    17    13    28     99    20
 [*,*,plate]:   FRA   DET   LAN   WIN   STL    FRE   LAF :=
	GARY     41    15    12    16    17     86     8
	CLEV     29     9    13     9    28     99    18
	PITT     26    14    17    13    31    104    20 ;
param fcost:     FRA DET LAN WIN STL FRE LAF :=
      GARY    3000 1200 1200 1200 2500 3500 2500
      CLEV    2000 1000 1500 1200 2500 3000 2200
      PITT    2000 1200 1500 1500 2500 3500 2200 ;
```

**[Figure 20-1b](#fig-20-1b):** Data for [Figure 20-1a](#fig-20-1a) (multmip1.dat).

```ampl
ampl: display sum {i in ORIG, j in DEST, p in PROD}
ampl?    vcost[i,j,p] * Trans[i,j,p];
sum{i in ORIG, j in DEST, p in PROD}
       vcost[i,j,p]*Trans[i,j,p] = 199500
```

```ampl
ampl: display Use;
Use [*,*]
:      DET    FRA    FRE    LAF    LAN    STL    WIN   :=
CLEV   1      0.44   0.52   0.36   0.64   0.88   0.32
GARY   0      0      1      0.24   0      1      0
PITT   0.84   1      0.36   1      0.16   1      0.28
;
```

As you can see, the total variable cost is the same as before, and Use assumes a variety of
fractional values. This solution tells us nothing new, and there is no simple way to convert
it into a good `integer` solution. An `integer` programming solver is essential to get any
practical results in this situation.

Switching to an appropriate solver, we find that the true optimum with fixed costs is
as follows:

```ampl
ampl: option solver cplex; solve;
CPLEX 8.0.0: optimal integer solution; objective 229850
295 MIP simplex iterations
19 branch-and-bound nodes
ampl: display {i in ORIG, j in DEST}
ampl?    sum {p in PROD} Trans[i,j,p];
sum{p in PROD} Trans[i,j,p] [*,*]
:      DET   FRA   FRE   LAF   LAN   STL  WIN  :=
CLEV   625   275     0   425   350   550  375
GARY     0     0   625     0   150   625    0
PITT   525   625   550   575      0  625    0
;
ampl: display Use;
Use [*,*]
:    DET FRA FRE LAF LAN STL WIN  :=
CLEV   1   1   0   1   1   1   1
GARY   0   0   1   0   1   1   0
PITT   1   1   1   1   0   1   0
;
```

Imposing the `integer` constraints has increased the total cost from $223,504 to $229,850;
but the number of unused routes has increased, to seven, as we had hoped.

**Zero-or-minimum restrictions**

Although the fixed-cost solution uses fewer routes, there are still some on which the
amounts shipped are relatively low. As a practical matter, it may be that even the variable
costs are not applicable unless some minimum number of tons is shipped. Suppose,
therefore, that we declare a parameter minload to represent the minimum number of
tons that may be shipped on a route. We could add a constraint to say that the shipments
on each route, summed over all products, must be at least minload:

```ampl
subject to Min_Ship {i in ORIG, j in DEST}: # WRONG
       sum {p in PROD} Trans[i,j,p] >= minload;
```

But this would force the shipments on every route to be at least minload, which is not
what we have in mind. We want the tons shipped to be either zero, or at least minload.
To say this directly, we might write:

```ampl
subject to Min_Ship {i in ORIG, j in DEST}: # NOT ALLOWED
       sum {p in PROD} Trans[i,j,p] = 0 or
       sum {p in PROD} Trans[i,j,p] >= minload;
```

But the current version of AMPL does not accept logical operators in constraints.

The desired zero-or-minimum restrictions can be imposed by employing the variables
Use[i,j], much as in the previous example:

```ampl
subject to Min_Ship {i in ORIG, j in DEST}:
       sum {p in PROD} Trans[i,j,p] >= minload * Use[i,j];
```

When total shipments from `i` to `j` are positive, Use[i,j] is 1, and Min_Ship[i,j]
becomes the desired minimum-shipment constraint. On the other hand, when there are no
shipments from `i` to j, Use[i,j] is zero; the constraint reduces to 0 >= 0 and has no
effect.

With these new restrictions and a minload of 375, the solution is found to be as follows
:

```ampl
ampl: model multmip2.mod;
ampl: data multmip2.dat;
ampl: solve;
CPLEX 8.0.0: optimal integer solution; objective 233150
279 MIP simplex iterations
17 branch-and-bound nodes
ampl: display {i in ORIG, j in DEST}
ampl?    sum {p in PROD} Trans[i,j,p];
sum{p in PROD} Trans[i,j,p] [*,*]
:      DET   FRA   FRE   LAF   LAN   STL   WIN  :=
CLEV   625   425   425     0   500   625     0
GARY     0     0   375   425      0  600     0
PITT   525   475   375   575      0  575   375
;
```

Comparing this to the previous solution, we see that although there are still seven unused
routes, they are not the same ones; a substantial rearrangement of the solution has been
necessary to meet the minimum-shipment requirement. The total cost has gone up by
about 1.4% as a result.

**Cardinality restrictions**

Despite the constraints we have added so far, origin PITT still serves 6 destinations,
while CLEV serves 5 and GARY serves 3. We would like to explicitly add a further
restriction that each origin can ship to at most maxserve destinations, where
maxserve is a parameter to the `model`. This can be viewed as a restriction on the size,
or cardinality, of a certain set. Indeed, it could in principle be written in the form of an
AMPL constraint as follows:

```ampl
subject to Max_Serve {i in ORIG}:    # NOT ALLOWED
       card {j in DEST:
              sum {p in PROD} Trans[i,j,p] > 0} <= maxserve;
```

Such a declaration will be rejected, however, because AMPL currently does not allow
constraints to use sets that are defined in terms of variables.

```ampl
set ORIG;                       # origins
set DEST;                       # destinations
set PROD;                       # products
param supply {ORIG,PROD} >= 0;  # amounts available at origins
param demand {DEST,PROD} >= 0;  # amounts required at destinations
       check {p in PROD}:
              sum {i in ORIG} supply[i,p] = sum {j in DEST} demand[j,p];
param limit {ORIG,DEST} >= 0;   # maximum shipments on routes
param minload >= 0;             # minimum nonzero shipment
param maxserve integer > 0;     # maximum destinations served
param vcost {ORIG,DEST,PROD} >= 0; # variable shipment cost on routes
var Trans {ORIG,DEST,PROD} >= 0;   # units to be shipped
param fcost {ORIG,DEST} >= 0;      # fixed cost for using a route
var Use {ORIG,DEST} binary;        # = 1 only for routes used
minimize Total_Cost:
       sum {i in ORIG, j in DEST, p in PROD} vcost[i,j,p] * Trans[i,j,p]
       + sum {i in ORIG, j in DEST} fcost[i,j] * Use[i,j];
subject to Supply {i in ORIG, p in PROD}:
       sum {j in DEST} Trans[i,j,p] = supply[i,p];
subject to Max_Serve {i in ORIG}:
       sum {j in DEST} Use[i,j] <= maxserve;
subject to Demand {j in DEST, p in PROD}:
       sum {i in ORIG} Trans[i,j,p] = demand[j,p];
subject to Multi {i in ORIG, j in DEST}:
       sum {p in PROD} Trans[i,j,p] <= limit[i,j] * Use[i,j];
subject to Min_Ship {i in ORIG, j in DEST}:
       sum {p in PROD} Trans[i,j,p] >= minload * Use[i,j];
```

**[Figure 20-2a](#fig-20-2a):** Multicommodity `model` with further restrictions (multmip3.mod).

Zero-one variables again offer a convenient alternative. Since the variables
Use[i,j] are 1 precisely for those destinations `j` served by origin i, and are zero otherwise,
we can write sum {j in DEST} Use[i,j] for the number of destinations
served by i. The desired constraint becomes:

```ampl
subject to Max_Serve {i in ORIG}:
       sum {j in DEST} Use[i,j] <= maxserve;
```

Adding this constraint to the previous `model`, and setting maxserve to 5, we arrive at
the mixed `integer` `model` shown in [Figure 20-2a](#fig-20-2a), with `data` shown in [Figure 20-2b](#fig-20-2b). It is
optimized as follows:

```ampl
set ORIG := GARY CLEV PITT ;
set DEST := FRA DET LAN WIN STL FRE LAF ;
set PROD := bands coils plate ;
param supply (tr):  GARY   CLEV   PITT :=
            bands    400    700    800
            coils    800   1600   1800
            plate    200    300    300 ;
param demand (tr):
               FRA  DET  LAN  WIN  STL  FRE  LAF :=
       bands   300  300  100   75  650  225  250
       coils   500  750  400  250  950  850  500
       plate   100  100    0   50  200  100  250 ;
param limit default 625 ;
param vcost :=
   [*,*,bands]:   FRA  DET  LAN  WIN  STL  FRE  LAF :=
          GARY     30   10    8   10   11   71    6
          CLEV     22    7   10    7   21   82   13
          PITT     19   11   12   10   25   83   15
   [*,*,coils]:   FRA  DET  LAN  WIN  STL  FRE  LAF :=
          GARY     39   14   11   14   16   82    8
          CLEV     27    9   12    9   26   95   17
          PITT     24   14   17   13   28   99   20
   [*,*,plate]:   FRA  DET  LAN  WIN  STL  FRE  LAF :=
          GARY     41   15   12   16   17   86    8
          CLEV     29    9   13    9   28   99   18
          PITT     26   14   17   13   31  104   20 ;
param fcost:     FRA DET LAN WIN STL FRE LAF :=
        GARY    3000 1200 1200 1200 2500 3500 2500
        CLEV    2000 1000 1500 1200 2500 3000 2200
        PITT    2000 1200 1500 1500 2500 3500 2200 ;
param minload := 375 ;
param maxserve := 5 ;
```

**[Figure 20-2b](#fig-20-2b):** Data for [Figure 20-2a](#fig-20-2a) (multmip3.dat).

```ampl
ampl: model multmip3.mod;
ampl: data multmip3.dat;
ampl: solve;
CPLEX 8.0.0: optimal integer solution; objective 235625
392 MIP simplex iterations
36 branch-and-bound nodes
ampl: display {i in ORIG, j in DEST}
ampl?    sum {p in PROD} Trans[i,j,p];
sum{p in PROD} Trans[i,j,p] [*,*]
:      DET   FRA   FRE   LAF   LAN   STL   WIN  :=
CLEV   625   375   550     0   500   550     0
GARY     0     0     0   400      0  625   375
PITT   525   525   625   600      0  625     0
;
```

At the cost of a further 1.1% increase in the `objective`, rearrangements have been made so
that GARY can serve WIN, bringing the number of destinations served by PITT down to
five.

Notice that this section's three `integer` solutions have served WIN from each of the
three different origins — a good example of how solutions to `integer` programs can jump
around in response to small changes in the restrictions.